# SignTalk AI: ST-GCN training on Google Colab (GPU)

**Pehle:** laptop pe `python scripts/pack_for_colab.py` chalao, aur bani hui `signtalk_colab.zip` ko Google Drive me `MyDrive/SignTalk/` folder me upload karo (purani zip ko replace karo).
Agar `scripts/record_signs.py` se apni recordings ki hain, to wo zip me khud aa jaati hain aur training me shaamil hoti hain.

**Colab me:** `Runtime -> Change runtime type -> T4 GPU` chuno, fir upar se ek-ek cell chalao (Shift+Enter).

Checkpoints aur results seedha Drive me save hote hain, to Colab disconnect ho jaaye tab bhi kuch nahi khoyega.

In [ ]:
# 1. GPU check: yahan 'Tesla T4' (ya koi GPU) dikhna chahiye. 'NO GPU' aaye to runtime type badlo.
import torch
print(torch.__version__, '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU')

In [ ]:
# 2. Google Drive mount (permission maangega, Allow karo)
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 3. Project unzip karo, aur experiments/ ko Drive se jodo taaki checkpoints wahin save hon.
#    Pichhli training ka folder hataya nahi jaata, experiments/stgcn_old_<time> naam se rakh diya jaata hai.
import time
DRIVE = '/content/drive/MyDrive/SignTalk'
OLD = f'stgcn_old_{time.strftime("%Y%m%d_%H%M")}'
!rm -rf /content/SignTalk-AI && mkdir -p /content/SignTalk-AI
!unzip -q $DRIVE/signtalk_colab.zip -d /content/SignTalk-AI
!mkdir -p $DRIVE/experiments $DRIVE/results
![ -d $DRIVE/experiments/stgcn ] && mv $DRIVE/experiments/stgcn $DRIVE/experiments/$OLD && echo "old run -> experiments/$OLD"
!ln -sfn $DRIVE/experiments /content/SignTalk-AI/experiments
%cd /content/SignTalk-AI
!pip -q install pyyaml
!wc -l data/manifests/*.csv

In [ ]:
# 4. Training (configs/stgcn.yaml, 80 epochs). Shuru me [DATA] lines batayengi ki apni recordings
#    mili ya nahi (har word ke train/val takes). Har epoch ki line yahin aayegi.
!python -u scripts/train_stgcn.py

In [ ]:
# 5. Test set pe evaluation -> results Drive me: MyDrive/SignTalk/results/stgcn
!python scripts/evaluate_stgcn.py --split test --output-dir $DRIVE/results/stgcn | grep -E 'Accuracy|F1'

In [ ]:
# 6. Live-jaisi checks: mirrored (ulte haath wale) signs, aur apni held-out recordings per word.
#    Purane model ke numbers: INCLUDE test 85.5%, mirrored 24.4%.
!pip install -q mediapipe   # imported by the realtime code the script uses; not in Colab by default
!python scripts/evaluate_robustness.py experiments/stgcn/checkpoints/best_checkpoint.pt | tee $DRIVE/results/stgcn/robustness.txt

## Laptop pe wapas lana

Drive se ye download karo:
- `MyDrive/SignTalk/experiments/stgcn/checkpoints/best_checkpoint.pt` -> laptop pe `D:\SignTalk-AI\best_checkpoint.pt` (project root me, abhi live model ko replace mat karo)
- `MyDrive/SignTalk/results/stgcn/` folder -> `D:\SignTalk-AI\results\stgcn_colab_v3\`

Fir Claude ko batao: wo purane model se compare karke decide karega ki naya model live (website + backend) me lagana hai ya nahi.